# Lesson 10 Lab — Taylor Importance: Ranking Channels by Loss Change

**Puzzle:** Can a small-norm channel still have a large effect on the loss?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Magnitude sees the parameter but not the data or objective. Taylor pruning uses the local product of an activation and its loss gradient to estimate how much removing a channel changes the loss. The approximation is cheap enough to rank many structures without a full retraining run for each one.


## 0. Predict before running

1. Predict whether L1 and Taylor produce identical rankings.
2. Write the first-order term for zeroing one activation channel.
3. Choose the correlation that validates each ranking against actual ablations.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A small classifier exposes one hidden activation tensor, its retained gradient, per-channel L1 weight scores, Taylor scores, and actual held-out loss increases from channel ablation.

- Taylor scores are objective- and data-dependent.
- Ablation loss change is the validation target for an importance ranking.
- First-order scores ignore interactions and distribution shift.


## 2. Derive the mechanism

If channel activation `h_c` is replaced by zero, first-order expansion gives `Delta L_c ≈ |∂L/∂h_c · (-h_c)|`, aggregated across samples and positions. Weight L1 instead ranks `sum |W_c|`. Taylor incorporates the current data and loss but remains local: interactions between channels and higher-order curvature are omitted. Correlation with actual one-channel ablation is the direct diagnostic for this toy problem.

### Mechanism at a glance

```mermaid
flowchart LR
  H["hidden activation h"] --> S["Taylor score |h × dL/dh|"]
  G["loss gradient dL/dh"] --> S
  S --> R["rank channels"]
  R --> A["held-out one-channel ablations"]
  A --> C["ranking correlation"]
  C --> J["joint-pruning validation"]
```

### Walk it step by step

1. **Capture the relevant activation.** Retain the hidden channel h and its gradient under a representative calibration loss.
2. **Compute the first-order score.** Aggregate the magnitude of h times dL/dh for each channel, with the sign policy stated explicitly.
3. **Validate the ranking.** Ablate channels one at a time on held-out data and compare predicted importance with actual loss increase.
4. **Recheck after joint pruning.** Independent first-order scores can fail when several interacting channels are removed together.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 10
LESSON_TITLE = 'Taylor Importance: Ranking Channels by Loss Change'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260818
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | channel ranking by outgoing/associated weight L1 magnitude |
| Candidate | channel ranking by absolute activation-gradient product |
| Held constant | trained toy classifier, calibration batch, held-out ablation batch, channel set, and loss |
| Measurements | Spearman correlation with actual loss increase, top-ranked channel, and loss deltas |
| Evidence | `numerical-model` |

**Experiment:** Compare L1 and Taylor channel rankings with exhaustive one-channel loss ablations on a held-out batch.


## 5. Read the experiment code

The notebook retains gradients on the hidden activation, performs one calibration backward pass, and aggregates `|h × grad|` per channel. It then runs controlled ablations on held-out inputs to construct the target ranking. The comparison measures ranking agreement rather than claiming a production pruning algorithm.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
n,d,h,c=1400,16,12,3
x=torch.randn(n,d,device=DEVICE); teacher=torch.randn(d,c,device=DEVICE); y=(x@teacher+0.3*torch.randn(n,c,device=DEVICE)).argmax(1)
train_x,cal_x,test_x=x[:900],x[900:1150],x[1150:]; train_y,cal_y,test_y=y[:900],y[900:1150],y[1150:]
model=nn.Sequential(nn.Linear(d,h),nn.ReLU(),nn.Linear(h,c)).to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=0.03)
for step in range(120):
    idx=torch.arange(step*64,step*64+64,device=DEVICE)%train_x.shape[0]; opt.zero_grad(); loss=F.cross_entropy(model(train_x[idx]),train_y[idx]); loss.backward(); opt.step()
model.eval(); hidden=F.relu(model[0](cal_x)); hidden.retain_grad(); logits=model[2](hidden); cal_loss=F.cross_entropy(logits,cal_y); model.zero_grad(); cal_loss.backward()
taylor=(hidden*hidden.grad).abs().mean(0).detach()
l1=(model[0].weight.abs().sum(1)+model[2].weight.abs().sum(0)).detach()
with torch.inference_mode():
    test_hidden=F.relu(model[0](test_x)); base_logits=model[2](test_hidden); base_loss=float(F.cross_entropy(base_logits,test_y).item()); actual=[]
    for channel in range(h):
        ablated=test_hidden.clone(); ablated[:,channel]=0; actual.append(float(F.cross_entropy(model[2](ablated),test_y).item()-base_loss))
actual_t=torch.tensor(actual)
metrics={
    "l1_spearman":spearman(l1.cpu(),actual_t),"taylor_spearman":spearman(taylor.cpu(),actual_t),
    "l1_top_channel":int(torch.argmax(l1).item()),"taylor_top_channel":int(torch.argmax(taylor).item()),
    "actual_top_channel":int(torch.argmax(actual_t).item()),"baseline_loss":base_loss,
    "l1_scores":l1.cpu().tolist(),"taylor_scores":taylor.cpu().tolist(),"actual_loss_increase":actual,
}
analysis=(
    f"Against exhaustive held-out ablations, L1 ranking had Spearman {metrics['l1_spearman']:.4f} and Taylor "
    f"had {metrics['taylor_spearman']:.4f}. Their top channels were {metrics['l1_top_channel']} and "
    f"{metrics['taylor_top_channel']}, while the largest actual loss increase came from channel "
    f"{metrics['actual_top_channel']}. The result tests one local ranking on one calibration batch."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| L1 Spearman | 0.867133 |
| Taylor Spearman | 0.895105 |
| L1 top channel | 5 |
| Taylor top channel | 5 |
| Actual top channel | 7 |
| Baseline loss | 0.126262 |


## 7. Interpret rather than merely print

Against exhaustive held-out ablations, L1 ranking had Spearman 0.8671 and Taylor had 0.8951. Their top channels were 5 and 5, while the largest actual loss increase came from channel 7. The result tests one local ranking on one calibration batch.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA experiment isolates a numerical mechanism. It is not a full paper reproduction, trained production model, or native sparse-kernel benchmark.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 10,
    "title": 'Taylor Importance: Ranking Channels by Loss Change',
    "environment": ENV,
    "evidence_label": 'numerical-model',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Taylor importance estimates local loss sensitivity; its value is established by held-out ablation agreement, not by the formula alone.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 10,
  "title": "Taylor Importance: Ranking Channels by Loss Change",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260818
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "l1_spearman": 0.8671328671328611,
    "taylor_spearman": 0.8951048951048889,
    "l1_top_channel": 5,
    "taylor_top_channel": 5,
    "actual_top_channel": 7,
    "baseline_loss": 0.12626215815544128,
    "l1_scores": [
      6.110829830169678,
      7.437592506408691,
      8.989200592041016,
      8.127813339233398,
      6.108587265014648,
      10.814108848571777,
      9.075263977050781,
      9.397420883178711,
      10.18946647644043,
      8.12380599975586,
      9.84946060180664,
      10.645395278930664
    ],
    "taylor_scores": [
      8.525817975169048e-05,
      0.00015044563042465597,
      0.0003051177482120693,
      0.0002408982982160523

## 9. Make the bounded decision

> Taylor importance estimates local loss sensitivity; its value is established by held-out ablation agreement, not by the formula alone.

**Acceptance/rollback:** Accept an importance metric only if its ranking is stable across representative batches and improves the declared quality-cost objective after pruning.

**Failure analysis:** One calibration batch can reverse scores, negative and positive first-order terms can cancel depending on aggregation, and simultaneous removal invalidates independent-channel estimates. Correlation on a tiny network does not establish ImageNet behavior.


## 10. Extend the evidence

Repeat across batches, compare signed, absolute, and second-order approximations, then prune several channels jointly and measure how ranking quality degrades with sparsity.

The full evidence boundary and references are in [`README.md`](README.md).
